In [ ]:
#| default_exp game/settlement

In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
#from fasthtml.jupyter import get_host
from fastlite import *
import fasthtml.components as fc
import httpx
import random
import pandas as pd
import threading
import numpy as np

??show

#| export
import logging

for logger_name in ("uvicorn", "uvicorn.error", "uvicorn.access"):
    uv_logger = logging.getLogger(logger_name)
    file_handler = logging.FileHandler('HexServer.txt')
    file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
    uv_logger.addHandler(file_handler)


show(Strong("I am strong text"))

sakura_headers = [Link(href='https://cdn.jsdelivr.net/npm/sakura.css/css/sakura.css', rel='stylesheet', type='text/css')]



app = FastHTML(hdrs = sakura_headers) # A FastHTML app, including the sakura CSS link in the headers
rt = app.route
server = JupyUvi(app) # Starts a server on port 8000 hosting the app
     


In [ ]:
#| export
from monsterui.all import *


In [ ]:
#| export
from HexMagic.game.globals import appRoutes,  webMe, globalStore, ensure_user,new_game_page, create_game, create_world, invalidate_cache, showUsers, logging

In [ ]:
from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame

In [ ]:
from HexMagic.primitives import HexTouchMap
from HexMagic.core import Terrain, DrainageBasins

In [ ]:
??appRoutes

In [ ]:
app, rt, hexGameServer  = appRoutes()

In [ ]:
!tail -10 base.text

In [ ]:
@rt
def hello(name: str): return P(f"Hello {name}!")

webMe(Div(
    H3("Say Hello"),
    Form(
        Input(placeholder="Your name...", id='name'),
        Button("Send"),
        hx_get=hello, hx_target="#result"
    ),
    Div(id='result'),
))

In [ ]:
#| export
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex, HexWrapper
from HexMagic.styles import StyleCSS,  SVGBuilder

from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord 
from HexMagic.primitives import HexGrid, HexPosition ,  HexRegion , windy_edge , unique_windy_edge ,HexLegend
from HexMagic.terrain import Terrain
from HexMagic.voronoi import generate_plate_terrain
Terrain.fromSeeds = generate_plate_terrain
from HexMagic.climate import ClimatePreset, Climate, TerraDemo
from HexMagic.geology import Geology, DrainageBasins, Watershed

In [ ]:
#| export
from HexMagic.game.data import ActiveGame, GameStorage, TerrainTemplate, Settlement, Piece, CountryFlag

In [ ]:
#| export
from HexMagic.database import ZoomResult, GeoStorageDebugger,GeoStorage, SaveResult, LoadResult, ChunkCover, User

## Helpers

In [ ]:
read_url(url="https://www.fastht.ml/docs/llms-ctx.txt")

In [ ]:
!cat ../../HexMagic/game/data.py

In [ ]:
!cat ../../HexMagic/plot/*.py

## Database

#| export
# Helper to ensure we have a user row
def ensure_user(session) -> int:
    if 'userid' not in session:
        session['userid'] = random.randint(0, 1_000_000)
    uid = session['userid']
    row = globalStore.db.execute("SELECT id FROM user WHERE id = ?", [uid]).fetchone()
    if not row:
        from datetime import datetime
        now = int(datetime.now().timestamp())
        globalStore.users.insert({
            'id': uid, 'username': f'player_{uid}', 'email': '', 'password': '',
            'created': now, 'sessionID': str(uid), 'activeWorld': 0
        })
    return uid



def new_game_page():
    templates = TerrainTemplate.maps  # {'bayArea': 'bayArea_map', ...}
    form = Form(
        Div(
            Label("Map Template", cls="label"),
            Select(
                *[Option(name, value=name) for name in sorted(templates.keys())],
                name="template_name", cls="select select-bordered w-full"
            ),
            cls="form-control"
        ),
        Div(
            Label("Kingdoms", cls="label"),
            Input(type="number", name="kingdoms", value="5", min="1", max="10",
                  cls="input input-bordered w-full"),
            cls="form-control"
        ),
        Div(
            Label("Lakes", cls="label"),
            Input(type="number", name="lakes", value="1", min="0", max="5",
                  cls="input input-bordered w-full"),
            cls="form-control"
        ),
        Div(
            Label("Hex Radius", cls="label"),
            Input(type="number", name="radius", value="25", min="10", max="40",
                  cls="input input-bordered w-full"),
            cls="form-control"
        ),
        Button("Create World", type="submit", cls="btn btn-primary mt-4"),
        action="/create_world", method="post",
        cls="card bg-base-200 shadow-lg p-6 space-y-4 max-w-md mx-auto"
    )
    return Titled("New Game",
        Div(
            H3("Choose Your World", cls="text-2xl font-bold text-center mb-6"),
            form,
            cls="flex flex-col items-center p-12"
        )
    )




#| export
@patch
def create_game(self: GameStorage, user_id, template_name="bayArea",
                kingdoms=5, lakes=1, radius=10) -> ActiveGame:
    logging.info(f"create_game: user={user_id} template={template_name}")
    tt = TerrainTemplate()
    terrain = getattr(tt, template_name)()
    
    terrain.carve_to_ocean(num_lakes=lakes)
    terrain.hexGrid.adjustRadius(radius)

    # GameBoard now creates cover + basins internally
    board = GameBoard(terrain, top_n=kingdoms)
    board.expand_kingdoms(max_rounds=50)
    
    # Save using the cover that GameBoard created
    board.cover.db = self
    board.cover.save(name=template_name)
    board.save(self, world_id=board.cover.ident)

    self.db.execute("UPDATE user SET activeWorld = ? WHERE id = ?",
                    [board.cover.ident, user_id])
    
    return ActiveGame(board=board, cover=board.cover, world_id=board.cover.ident)


_game_cache = {}
_cache_lock = threading.Lock()

@patch
def active_board(self: GameStorage, user_id) -> ActiveGame:
    with _cache_lock:
        if user_id in _game_cache:
            logging.info(f"active_board: cache hit for user {user_id}")
            return _game_cache[user_id]
    
    logging.info(f"active_board: cache miss, building for user {user_id}")
    row = self.db.execute("SELECT activeWorld FROM user WHERE id = ?", [user_id]).fetchone()
    if not row or not row[0]:
        return None
    
    world_id = row[0]
    try:
        # gameboard() now loads cover internally and reuses cover.basin
        board = self.gameboard(world_id)
        active = ActiveGame(board=board, cover=board.cover, world_id=world_id)
        
        with _cache_lock:
            _game_cache[user_id] = active
        
        return active
    except Exception as e:
        logging.error(f"active_board: FAILED: {e}", exc_info=True)
        return None


## Common Routes

@rt
def create_world(session, template_name: str, kingdoms: int = 5,
                 lakes: int = 1, radius: int = 10):
    uid = ensure_user(session)
    globalStore.create_game(uid, template_name=template_name,
                            kingdoms=kingdoms, lakes=lakes, radius=radius)
    logging.info(f"create world redirecting")
    return RedirectResponse('/game', status_code=303)


@rt
def create_world(session, template_name: str, kingdoms: int = 5,
                 lakes: int = 1, radius: int = 10):
    uid = ensure_user(session)
    logging.info(f"create_world: session keys={list(session.keys())}")
    invalidate_cache(uid)  # <-- clear old game
    globalStore.create_game(uid, template_name=template_name,
                            kingdoms=kingdoms, lakes=lakes, radius=radius)
    return RedirectResponse('/game', status_code=303)

_game_cache = {}
_cache_lock = threading.Lock()

def invalidate_cache(user_id):
    """Call this when creating a new game or the board changes."""
    with _cache_lock:
        _game_cache.pop(user_id, None)


### Kingdoms

In [ ]:
@rt
def select_kingdom(session, country_id: int):
    """Handle kingdom hex button click from left panel."""
    return kingdom(session, int(country_id))


@rt
def kingdom_list_panel(session):
    """ This gives a list of kingdoms to select """
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return P("No active game")

    board = active.board
    terrain = board.terrain
    current_radius = int(terrain.hexGrid.radius)

    # Build styled hex buttons for each kingdom
    kingdom_hexes = []
    kingdom_values = []
    for k in board.kingdoms:
        style = StyleCSS(f"k_{k.countryId}",
                         fill=k.flag.primary,
                         stroke=k.flag.comp,
                         stroke_width=2)
        h = Hex(radius=25, center=MapCord(0, 0), style=style,
                label=k.countryName or f"Kingdom {k.countryId}")
        kingdom_hexes.append(h)
        kingdom_values.append(k.countryId)

    kingdom_buttons = HexButtonGroup(
        kingdom_hexes,
        name="country_id",
        values=kingdom_values,
        hx_post="/select_kingdom",
        hx_target="#map",
        hx_swap="outerHTML",
        direction="column",
        height=45,
        font_size=11,
        char_width=7,
    )

    # Radius slider
    radius_control = Div(
        P("Hex Size", cls=TextPresets.bold_sm),
        Input(type="range", name="radius", id="radius-slider",
              min="10", max="40", value=str(current_radius),
              hx_get="/showMap", hx_target="#map",
              hx_trigger="change", cls="uk-range w-full"),
        P(f"{current_radius}px", id="radius-label", cls=TextPresets.muted_sm),
        Script("""
            me('#radius-slider').on('input', ev => {
                me('#radius-label').textContent = ev.target.value + 'px';
            });
        """),
        cls="space-y-2"
    )

    # Size hex background to fit content
    n = len(board.kingdoms)
    panel_h = max(400, 120 + n * 55 + 120)  # kingdoms + controls
    panel_w = 240

    panel_content = Div(
        P("Kingdoms", cls=TextPresets.bold_sm),
        kingdom_buttons,
        Divider(),
        radius_control,
        Divider(),
        A("🌍 New Game", href="/new_game", cls="btn btn-outline btn-sm"),
        cls="flex flex-col items-center gap-2 p-4",
        style="position: relative; z-index: 1;"
    )

In [ ]:
@rt
def kingdom_hex_clicked(session, hex_id: int, country_id: int):
    uid = ensure_user(session)
    logging.info(f"kingdom_hex_clicked: uid={uid}")
    active = globalStore.active_board(uid)
    if not active:
        return RedirectResponse('/', status_code=303)

    board = active.board
    terrain = board.terrain

    result = globalStore.kingdom_detail(active.world_id, country_id, active.cover)
    coarse_idx = result.mapper(hex_id)

    countries = terrain.fields.get("country")
    if countries is None or coarse_idx < 0 or coarse_idx >= len(countries):
        return RedirectResponse('/showMap', status_code=303)

    owner = int(countries[coarse_idx])

    if owner == country_id or owner <= 0:
        # Zoom out to world view
        return (
            HtmxResponseHeaders(push_url="/game"),
            showMap(session),
            Div("Zooming out to world view", id="debug-panel", hx_swap_oob="true")
        )
    else:
        # Zoom into the other kingdom
        k = next((k for k in board.kingdoms if k.countryId == owner), None)
        return (
            HtmxResponseHeaders(push_url=f"/kingdom/{owner}"),
            kingdom(session, owner),
            Div(f"Zooming into {k.countryName if k else f'Kingdom {owner}'}",
                id="debug-panel", hx_swap_oob="true")
        )


In [ ]:
@rt("/kingdom/{id}")
def kingdom(session, id: int):
    uid = ensure_user(session)
    logging.info(f"kingdom: uid={uid}")
    active = globalStore.active_board(uid)
    if not active:
        logging.info(f"kingdom: not active")
        return RedirectResponse('/', status_code=303)

    board = active.board
    terrain = board.terrain

    result = globalStore.kingdom_detail(active.world_id, id, active.cover)

    zoomed = result.terrain
    zoomed.hexGrid.adjustRadius(terrain.hexGrid.radius)
    zoomed.hexGrid.builder.layers = []
    zoomed.colorMap()
    zoomed.hexGrid.update()

    c2f = result.invert_mapper()

    builder = zoomed.hexGrid.builder
    
    zoomed.terrainCream()

    builder.adjust("borders", board.countries_overlay(zoomed, c2f))
    builder.adjust("names", board.names_overlay(zoomed, c2f))

    if result.basins:
        builder.adjust("water", result.basins.draw_watersheds())

    k = next((k for k in board.kingdoms if k.countryId == id), None)
    title = k.countryName if k else f"Kingdom {id}"
    logging.info(f"kingdom {title} for uid={uid} is {id}")

    on_click = lambda grid, index: {
        "hx-post": "/kingdom_hex_clicked",
        "hx-vals": json.dumps({"hex_id": index, "country_id": id}),
        "hx-target": "#map",
    }


    return Div(
        Div(
            A("← Back to World", hx_get="/showMap", hx_target="#map",
              cls="btn btn-sm btn-outline"),
            H4(title, cls="text-lg font-bold"),
            cls="flex items-center gap-4 p-2"
        ),
        HexTouchMap(zoomed.hexGrid, on_click=on_click, cls="w-full h-full"),
        id="map"
    )

In [ ]:
@rt("/piece_detail/{id}")
def piece_detail(session, id: str):
    """Placeholder for piece detail — right panel."""
    return Div(
        H4(f"🧑 Piece", cls="font-bold"),
        P(f"ID: {id[:12]}…", cls=TextPresets.muted_sm),
        Divider(),
        P("Actions and stats coming soon", cls="opacity-50 text-sm"),
        cls="space-y-2"
    )


## Overlays

In [ ]:
# ── Overlay session helpers ──────────────────────────────────

DEFAULT_OVERLAYS = {'settlements', 'names', 'elevation', 'rivers'}

OVERLAY_CONFIG = {
    'cream':       {'label': '🏔️ Terrain',       'fill': '#F5DEB3', 'stroke': '#8B7355'},
    'rivers':      {'label': '🌊 Rivers',        'fill': '#4169E1', 'stroke': '#1E3A6E'},
    'names':       {'label': '📝 Names',         'fill': '#4A90D9', 'stroke': '#2C5F8A'},
    'settlements': {'label': '🏰 Settlements',   'fill': '#DAA520', 'stroke': '#8B6914'},
    'temperature': {'label': '🌡️ Temperature',   'fill': '#E8734A', 'stroke': '#A34520'},
    'climate':     {'label': '🌿 Climate',       'fill': '#7A9B76', 'stroke': '#4A6B46'},
    'watersheds':  {'label': '💧 Watersheds',    'fill': '#6B8EC4', 'stroke': '#3A5E94'},
    'flow':        {'label': '🧭 Flow',          'fill': '#555555', 'stroke': '#333333'},
    'elevation':   {'label': '⛰️ Elevation',     'fill': '#9D8B73', 'stroke': '#6D5B43'},
}


def get_overlays(session) -> set:
    raw = session.get('overlays')
    if raw is None: return set(DEFAULT_OVERLAYS)
    return set(raw)

def set_overlays(session, overlays: set):
    session['overlays'] = list(overlays)


In [ ]:
@patch
def apply_overlays(zoomed: Terrain, result: ZoomResult, board: GameBoard,
                   c2f: dict, overlays: set, settle_attrs: dict = None,level:int=3):
    """Configure terrain builder layers based on active overlay set.
    
    Pure helper — no session, no redirects, no route logic.
    """
    builder = zoomed.hexGrid.builder

    # Always show borders
    builder.adjust("borders", board.countries_overlay(zoomed, c2f))

    if 'cream' in overlays:
        zoomed.terrainCream()

    if 'elevation' in overlays:
        builder.adjust("elevation", zoomed.elevation_borders())

    if 'temperature' in overlays:
        try:
            builder.adjust("temperature", zoomed.render_icon_temperature())
        except ValueError:
            pass  # fields not computed

    if 'climate' in overlays:
        try:
            builder.adjust("climate", zoomed.dottedClimate())
        except ValueError:
            pass

    if 'watersheds' in overlays and result.basins:
        builder.adjust("watersheds", result.basins.dotted_watershed_overlay())

    if 'rivers' in overlays and result.basins:
        builder.adjust("water", result.basins.draw_watersheds())

    if 'flow' in overlays:
        try:
            builder.adjust("flow", zoomed.flow_diagram())
        except Exception:
            pass

    if 'names' in overlays:
        builder.adjust("names", board.names_overlay(zoomed, c2f))

    if 'settlements' in overlays and settle_attrs:
        builder.adjust("settlements",
                       board.settlementTouchOverlay(zoomed, c2f, attrs=settle_attrs))


In [ ]:
@rt("/toggle_overlay/{id}")
def toggle_overlay(session, id: str, overlay: str):
    """Toggle a single overlay on/off and re-render the map + controls."""
    current = get_overlays(session)
    if overlay in current:
        current.discard(overlay)
    else:
        current.add(overlay)
    set_overlays(session, current)

    # Re-render both the map and the controls bar
    map_partial = settlement_map(session, id)
    controls_partial = Div(
        settlement_controls(session, id),
        id="map-controls",
        hx_swap_oob="true"
    )
    return map_partial, controls_partial


## draw some

def flag_hex(flag, radius=25):
    """Hex with flag pattern fill, built via SVGBuilder."""
    pad = 4
    size = radius * 2 + pad * 2
    center = MapCord(size / 2, size / 2)

    builder = SVGBuilder()
    builder.width = size
    builder.height = size

    # Register flag pattern as a fill
    pat_name = f"fh_{flag.name}"
    builder.add_definition(flag.flagPattern(pat_name, scale=0.08))
    pat_style = StyleCSS(pat_name, fill=f"url(#{pat_name})",
                         stroke=flag.comp, stroke_width=3)
    builder.add_style(pat_style)

    h = Hex(radius=radius, center=center, style=pat_style)
    builder.adjust("hex", h.svg())

    return Div(NotStr(builder.xml()))


In [ ]:
??Terrain.elevation_borders

In [ ]:
??DrainageBasins.dotted_watershed_overlay

In [ ]:
??Terrain.render_icon_temperature

In [ ]:
??Terrain.flow_diagram

In [ ]:
??Terrain.dottedClimate

So there are a few more overlays we want to add

can you add all of them OVERLAY_CONFIG with nice emoji like the others

In [ ]:
GameStorage.settlement_from_id??

Lets think about the map panel first. We haven't yet built the higher panes so there is nothing to go back to. 

we will need something like
@rt("/settlment{id}")
def settlment(session, id: int):
    uid = ensure_user(session)
    logging.info(f"settlment: uid={uid}")
    active = globalStore.active_board(uid)
    # we get the settlement, kinddom this is from
    # we get terrain a certain radius around the settlment.
    # we build a touch layer with pieces on top. for right now we don't have any pieces so the touch layer needs to be just the settlement
    # we have a HexTouchLayer, but this needs to go underneath piece touch layer (with the hope that clicks don't hit everything in the stack 

I want to see about rings around the scope. for kingodms we generate a map based upon a HexRRegion. We would try to create a region to get our map back. I don't know if it makes sense to have a
@rt("/settlement/{id}&{ringlenght}") or how you do multiple parameters into a page. we would have a default ringsize if none is provided

Yes for the UUID. What next to think about?

In [ ]:
#!cat ../../HexMagic/plot/*.py

We have a very rich region system

zoom_region_fast because we want the details

## `zoom_region_fast` — Deep Dive

This is the **main pipeline** for the question: *"I have a HexRegion on the coarse grid — give me a high-resolution Terrain covering just that region, with weather and watersheds."*

It orchestrates every piece we've built so far into a 10-step pipeline that stays in numpy-land as long as possible, only constructing the expensive `HexGrid` geometry once at the very end.

### The pipeline at a glance

```
HexRegion (coarse)
    │
    ▼
┌─────────────────┐
│ 1. Bounding box  │  O(|region|)
│ 2. Chunk refs    │  O(bbox area / spacing²)
│ 3. Stitch chunks │  O(chunks × chunk_size)  ← numpy block copies
│ 4. Build mapper  │  O(1) closure creation
│ 5. Region filter │  O(merged_size)  ← identifies fine hexes in region
│ 6. Crop proxy    │  O(cropped_size) ← numpy 2D slice
│ 7. Wrap mapper   │  O(1) closure wrapping
│ 8. Build Terrain │  O(cropped_size) ← HexGrid constructed ONCE here
│ 9. Project weather│  O(valid_fine_hexes) ← vectorized numpy scatter
│10. Project sheds │  O(valid_fine_hexes) ← dict grouping + lookup
└─────────────────┘
    │
    ▼
ZoomResult(terrain, basins, mapper, chunks_loaded)
```

### Step 1 — Bounding box with padding

Given a `HexRegion` (a set of coarse hex indices), we find the axis-aligned bounding box in row/col space:

$$r_{\min}, r_{\max}, c_{\min}, c_{\max} = \text{region\_bounding\_box}(\text{region})$$

Then we **pad** by `halo_rings + 1` in every direction (clamped to grid bounds). The padding ensures that when we zoom, fine hexes at the region boundary have enough neighbor data for interpolation and drainage flow:

$$r_{\min}' = \max(0,\; r_{\min} - p) \qquad r_{\max}' = \min(N_r - 1,\; r_{\max} + p)$$

where $p = \text{halo\_rings} + 1$.

**Why halo_rings?** Each chunk already has a halo of extra hexes for seamless stitching. The `+1` ensures we don't get edge artifacts on the region boundary itself.

### Step 2 — Chunk discovery

`bbox_to_chunk_refs` samples the padded bounding box at `spacing = 2 × rings` intervals and maps each sample point to its chunk via `world_to_chunk`:

```
Padded bbox on coarse grid:
┌─────────────────────────┐
│  ·  ·  ·  ·  ·  ·  ·   │   · = sample points at spacing intervals
│                         │
│  ·  ·  ·  ·  ·  ·  ·   │   Each · → world_to_chunk → ChunkRef
│                         │
│  ·  ·  ·  ·  ·  ·  ·   │   + corners checked explicitly
└─────────────────────────┘
```

The result is a `set[ChunkRef]` — deduplicated because multiple sample points can land in the same chunk.

**Corner check:** The loop at `spacing` intervals can miss chunks that *just barely* overlap a corner. The explicit corner sampling catches those.

### Step 3 — Stitch chunks (the heavy lift)

`stitch_chunks_fast` does:

1. For each `ChunkRef`, look up the cached zoomed chunk as a `DataProxy`
2. On cache miss, call `load_or_generate_chunk` (runs the actual zoom computation), then reload as `DataProxy`
3. Compute the merged grid dimensions from the range of coarse positions × scale
4. Copy each chunk's 2D elevation/watershed arrays into the merged array at its offset

The key equations for merged dimensions:

$$H_{\text{merged}} = (r_{\max}^{\text{coarse}} - r_{\min}^{\text{coarse}}) \times \text{scale} + H_{\text{chunk}}$$
$$W_{\text{merged}} = (c_{\max}^{\text{coarse}} - c_{\min}^{\text{coarse}}) \times \text{scale} + W_{\text{chunk}}$$

And the placement offset for each chunk:

$$\text{row\_off}^{(k)} = (r_k^{\text{coarse}} - r_{\min}^{\text{coarse}}) \times \text{scale}$$
$$\text{col\_off}^{(k)} = (c_k^{\text{coarse}} - c_{\min}^{\text{coarse}}) \times \text{scale}$$

This is all numpy 2D slice assignment — no per-hex loops.

### Step 4 — Build the mapper (on uncropped proxy)

This is the `build_fine_to_coarse_mapper` from the previous deep dive. The critical point: it's built on the **uncropped** merged proxy, because we need it in Step 5 before we know what to crop.

### Step 5 — Identify the region's fine hexes

This is the step that connects the coarse `HexRegion` to the fine grid. For every valid fine hex:

$$\text{coarse\_idx} = \text{mapper}(\text{fine\_idx})$$

If `coarse_idx ∈ region.hexes`, this fine hex "belongs" to the region.

```
Coarse region (shaded):          Fine merged grid:
┌──┬──┬──┬──┬──┐                 ┌──┬──┬──┬──┬──┬──┬──┬──┬──┬──┐
│  │  │░░│░░│  │   mapper        │  │  │  │  │▓▓│▓▓│▓▓│▓▓│  │  │
│  │  │░░│░░│  │   ←────────     │  │  │  │  │▓▓│▓▓│▓▓│▓▓│  │  │
│  │  │░░│  │  │                 │  │  │  │  │▓▓│▓▓│  │  │  │  │
│  │  │  │  │  │                 │  │  │  │  │  │  │  │  │  │  │
└──┴──┴──┴──┴──┘                 └──┴──┴──┴──┴──┴──┴──┴──┴──┴──┘

                                  ▓▓ = fine hexes where mapper → region hex
```

Three types of fine hexes get marked **invalid** here:
- Already in `invalidRegion` (from stitching)
- Elevation ≤ −99.0 (deep water sentinel)
- Mapper returns −1 (outside any chunk)

This is the most expensive loop — `O(merged_size)` — but it's pure Python integer ops, no geometry.

### Step 6 — Crop the proxy

The merged proxy can be much larger than the region. Imagine a region that's 5 coarse hexes in a vertical strip — the merged proxy might be 4 chunks wide but we only need a narrow column.

$$\text{crop\_padding} = \max(4,\; \text{scale} \times (\text{halo\_rings} + 1))$$

The crop finds the bounding box of `region_fine_indices`, pads it, and extracts a numpy 2D slice:

```
Before crop (merged proxy):      After crop:
┌────────────────────────┐        ┌──────────┐
│                        │        │          │
│       ┌──────┐         │   →    │ ┌──────┐ │
│       │region│         │        │ │region│ │
│       └──────┘         │        │ └──────┘ │
│                        │        │          │
└────────────────────────┘        └──────────┘
    50×80 = 4000 cells              20×30 = 600 cells
```

**Bail-out:** If cropping would save less than 10% of cells, we skip it (the overhead isn't worth it).

The function returns the crop offsets `(trim_top, trim_left)` needed to translate cropped coordinates back to merged coordinates.

### Step 7 — Wrap the mapper

Since the proxy was cropped, fine indices shifted. `make_cropped_mapper` wraps the original mapper:

$$\text{merged\_row} = \text{cropped\_row} + \text{trim\_top}$$
$$\text{merged\_col} = \text{cropped\_col} + \text{trim\_left}$$
$$\text{merged\_idx} = \text{merged\_row} \times W_{\text{old}} + \text{merged\_col}$$

Then feeds `merged_idx` to the original uncropped mapper. If no cropping happened (trim = 0), the original mapper is returned as-is.

### Step 8 — Build Terrain ONCE

`proxy_to_terrain(cropped_proxy)` is the **only** place where `HexGrid` geometry gets constructed. This is the payoff of the entire DataProxy design:

- `stitch_chunks_fast` → numpy only
- `crop_proxy_to_region` → numpy only
- Steps 5-7 → integer arithmetic only
- **Here** → create hex objects, compute pixel positions, etc.

We do it on the **cropped** proxy, so the HexGrid is as small as possible.

### Step 9 — Project weather (vectorized)

Weather lives on the coarse terrain. We need it on the fine terrain. The mapper gives us the correspondence, and we vectorize it:

```python
fine_arr   = [  0,   1,   2,   5,   6, ...]   # fine indices (valid)
coarse_arr = [ 42,  42,  42,  43,  43, ...]   # their coarse parents
```

For each weather field (temperature, precipitation, etc.):

$$\text{fine\_field}[\text{fine\_arr}] = \text{coarse\_field}[\text{coarse\_arr}]$$

This is a single numpy **fancy-index scatter** — no Python loop over hexes. Every fine hex in a `scale × scale` block gets the same coarse value (nearest-neighbor projection).

**Fallback:** If temperature or precipitation still isn't populated (e.g. the coarse terrain didn't have them), it calls `merged_terrain.compute_weather()` to generate from scratch.

### Step 10 — Project watersheds

We group the fine→coarse mapping into `fine_regions`:

$$\text{fine\_regions}[c] = \{f \;|\; \text{mapper}(f) = c\}$$

This is exactly the dict that `project_watersheds` expects — it assigns each fine hex the watershed ID of its coarse parent from the `DrainageBasins`.

The coarse `DrainageBasins` is lazily computed once (Step 3's `self.basin` check) and then reused for all subsequent zooms.

---

### Worked example

Setup:
- `cover` with `rings=3`, `halo_rings=1`, coarse grid `20×20`
- Region = 6 coarse hexes forming an L-shape
- `scale = 2`

```
Step 1:  bbox = rows [4,8], cols [6,9]
         padded = rows [2,10], cols [4,11]  (padding = 2)

Step 2:  At spacing=6, samples hit chunks (0,0,0) and (1,0,-1)
         → 2 ChunkRefs

Step 3:  Each chunk zooms to 14×14 fine hexes
         Merged proxy = 14×28 = 392 cells  (chunks side by side)

Step 4:  Mapper closure built (captures chunk_offsets, scale=2, etc.)

Step 5:  Loop over 392 fine cells:
         - 168 map to region hexes → region_fine_indices
         - 180 map to non-region hexes → ignored
         - 44 invalid (water/edge)

Step 6:  region_fine_indices bbox = rows [4,17], cols [4,17]
         Crop to 18×18 = 324 cells (saves 17%)
         trim_top=4, trim_left=4

Step 7:  Mapper wrapped: cropped_idx → +4 to both row,col → original mapper

Step 8:  HexGrid(18, 18, radius=5) built — 324 hexes with geometry

Step 9:  temperature[fine_arr] = coarse_temp[coarse_arr]  (one numpy op per field)

Step 10: fine_regions = {42: {0,1,2,3}, 43: {4,5,6,7}, ...}
         Each set gets its parent's watershed_id
         → ZoomResult with 168 meaningful fine hexes
```

### Performance characteristics

| Step | Cost | Dominates when... |
|------|------|-------------------|
| 3. Stitch | O(chunks × chunk_size) | Many chunks, cache cold |
| 5. Region filter | O(merged_size) | Large merged proxy |
| 8. Build Terrain | O(cropped_size) | Large region |
| 9. Weather | O(valid_fine × fields) | Many weather fields |

The cache-cold case (first zoom) is dominated by Step 3 — actually computing the zoomed terrain. Subsequent zooms hit the cache and Steps 5-9 dominate, which are all numpy-fast.

### The returned ZoomResult

```python
@dataclass
class ZoomResult:
    terrain: Terrain          # Full Terrain with HexGrid, elevations, weather fields
    basins: DrainageBasins    # Watersheds projected from coarse → fine
    mapper: Callable          # cropped_fine_idx → coarse_idx (for further lookups)
    chunks_loaded: int        # How many chunks were involved
```

The `mapper` is returned so callers can do their own fine→coarse lookups (e.g., "which biome does this fine hex inherit?" or "what political territory is it in?"). It's the cropped version, so it works directly with indices into `terrain.hexGrid`.

## Settlement View — Project Plan

### What we have
- ✅ `Settlement.region(grid, rings)` — builds HexRegion around settlement
- ✅ `zoom_region_fast` — high-res terrain from any HexRegion
- ✅ `HexTouchMap` — interactive SVG hex map with click handlers
- ✅ `settlement_from_id` / `kingdom_from_db` — DB loading
- ✅ Flag pieces with touch event support (`flag.kingPiece(...)`)
- ✅ Country/name overlay system
- ✅ Trade route data model (`TradeRoute`)

### Phase 1 — Core settlement map
1. **`Settlement.zoom(cover, rings)`** — patch that calls `zoom_region_fast` on `self.region()`, returns `ZoomResult`
2. **`@rt("/settlement/{id}")`** — route that loads settlement, zooms terrain, renders `HexTouchMap` with settlement marker overlay
3. Basic styling — settlement circle marker at center, kingdom borders if visible

### Phase 2 — Left panel
4. **Breadcrumbs** — `🌍 World → Kingdom Name → Settlement Name` with `hx-get` links back to each level
5. **City stats card** — name, size, health, citizen count, owner kingdom
6. **Piece list** — each citizen piece as a clickable item (flag hex buttons or list items)

### Phase 3 — Piece interaction
7. **Flag pieces on map** — render `flag.kingPiece()` at each citizen's location hex with HTMX touch attrs
8. **Piece detail panel** — when a piece is clicked, show stats + action buttons in a lower/right panel
9. **Piece touch layer** — SVG layer above hex touch layer, `pointer-events: none` on decorative layers

### Phase 4 — Trade routes
10. **Trade route overlay** — draw `TradeRoute.path` as styled `MapPath` on the zoomed terrain
11. **Route interaction** — click a route to see cost/destination info

### Phase 5 — Full page shell
12. **Shell template** — shared 3-column layout (left panel, map, right panel) reused across all modes
13. **HTMX vs full page** — `htmx` param check so direct URL hits render full page, HTMX requests return partials
14. **`hx-push-url`** — URL updates as you drill down: `/game` → `/kingdom/3` → `/settlement/abc123`

### Open questions
- Do pieces move on the settlement map, or is that a future turn-based mechanic?
- Should the settlement zoom level be adjustable (rings slider) or fixed?
- Where do piece *actions* live (harvest, settle, split) — in the detail panel or on the map?

I think these should be on settlement and we are lucky it has its db where we can make calls to

Instead of spiral you might want to consider
```
@patch
def indices_in_range(self: HexGrid, index: int, distance: int) -> np.ndarray:
    """Get all valid hex indices within `distance` steps of `index`, purely via numpy."""
    q = np.arange(-distance, distance + 1)
    qq, rr = np.meshgrid(q, q)
    ss = -qq - rr
    mask = np.abs(ss) <= distance
    coords = np.stack([qq[mask], rr[mask], ss[mask]], axis=-1)
    indices = self.hexpositions_to_indices(coords, origin_index=index)
    return indices[indices >= 0]
```
I am in the process of adding it


In [ ]:
@patch
def region(self: Settlement, grid: HexGrid, rings: int = 5) -> HexRegion:
    indices = grid.indices_in_range(self.location, rings)
    return HexRegion(hexes=set(indices), hexGrid=grid)


Lets do this tomorrow. For now just right up a project plan for this settlement idea of where we are headed

In [ ]:
showUsers()

In [ ]:
dummySession= {'userid': 64801 }

In [ ]:
def showPieces():
    users_df = pd.DataFrame(globalStore.pieces())
    print("pieces:")
    print(users_df)

In [ ]:
showPieces()

In [ ]:
def showActiveSettlements(session):
    uid = ensure_user(session)
    logging.info(f"showSettlement: uid={uid}")
    active = globalStore.active_board(uid)
    ret = []
    for country in active.board.kingdoms:
        ret.extend( country.settlements)
    return ret


In [ ]:
showActiveSettlements(dummySession)

In [ ]:
def listSettlements():
    users_df = pd.DataFrame(globalStore.settlements())
    print("pieces:")
    print(users_df)

In [ ]:
listSettlements()

In [ ]:
dummySettlment = "e03264bc-9fa4-4fc6-9ca7-6e97e6f088e0"

In [ ]:
def showSettlement(session, settlement_id:str):
    uid = ensure_user(session)
    logging.info(f"showSettlement: uid={uid}\tsettlement_id={settlement_id}")
    active = globalStore.active_board(uid)
    if not active:
        logging.info(f"kingdom: not active")
        return RedirectResponse('/', status_code=303)

    try:
        place = globalStore.settlement_from_id(settlement_id)

        #board = active.board
        terrain = active.board.terrain
        region = place.region(terrain.hexGrid)
        logging.info(f"showSettlement[{settlement_id}]: {len(region.hexes) }")
        if len(region.hexes) == 0:
            return None
        # Ensure cover is wired to storage for chunk caching
        #cover.db = self

        # zoom_region reuses cover.basin — no O(n²) recompute
        return active.cover.zoom_region_fast(region,  compute_weather=True)
    except Exception as e:
        logging.error(f"showSettlement FAILED: {e}", exc_info=True)
        return None

In [ ]:
zResult = showSettlement(dummySession,dummySettlment)

In [ ]:
zResult

In [ ]:
aBoard = globalStore.active_board(dummySession['userid']).board
aBoard

In [ ]:
@patch
def project_region(region: HexRegion, target_grid: HexGrid, c2f: dict = None) -> HexRegion:
    """Project a coarse region onto a target grid via c2f mapping.
    If c2f is None (identity), returns the region unchanged."""
    if c2f is None:
        return region
    fine_hexes = set()
    for coarse_idx in region.hexes:
        if coarse_idx in c2f:
            fine_hexes.update(c2f[coarse_idx])
    return HexRegion(hexes=fine_hexes, hexGrid=target_grid)


def _map_point(coarse_idx: int, c2f: dict = None) -> int:
    """Map a single coarse index to a fine index (picks middle representative).
    If c2f is None (identity), returns coarse_idx unchanged."""
    if c2f is None:
        return coarse_idx
    fs = c2f.get(coarse_idx)
    if not fs:
        return -1
    return fs[len(fs) // 2]

In [ ]:
@patch
def settlementTouchOverlay(self: GameBoard, terrain: Terrain = None,
                      c2f: dict = None, scale: float = 2.0,attrs= None) -> str:
    """Settlement markers using king pieces with flag patterns.

       Place a chess king at center. attrs: dict of HTML/HTMX attributes for interactivity.
    For instance:
    Then usage looks like:
    
flag.kingPiece(center, scale=1.5, attrs={
    'hx-get': '/piece/king/clicked',
    'hx-target': '#game-panel',
    'hx-swap': 'innerHTML',
    'data-piece': 'king',
    'data-player': flag.name
})

    
    """
    terrain = terrain or self.terrain
    grid = terrain.hexGrid
    num_hexes = len(grid.hexes)

    overlay = ""
    for i, country in enumerate(self.kingdoms):
        if country.flag is None:
            continue

        # Register flag pattern
        pat_name = f"settle_pat_{country.countryId}"
        pat = country.flag.flagPattern(pat_name, scale=0.1)
        grid.builder.add_definition(pat)
        pat_fill = f"url(#{pat_name})"

        for j, s in enumerate(country.settlements):
            if s.location is None:
                continue
            local_idx = _map_point(s.location, c2f)
            if local_idx < 0 or local_idx >= num_hexes:
                continue
            coords = grid.hexes[local_idx].center
            piece_id = f"king_{country.countryId}_{j}"
            overlay += country.flag.kingPiece(
                MapCord(coords.x, coords.y),
                scale=scale, piece_id=piece_id, fill=pat_fill, attrs=attrs
            )

    return overlay


In [ ]:
zTerr = zResult.terrain
zTerr.hexGrid.builder.layers = []
zTerr.colorMap()
zTerr.hexGrid.update()
c2f = zResult.invert_mapper()

borders = aBoard.countries_overlay(zResult.terrain, c2f)
names   = aBoard.names_overlay(zResult.terrain, c2f)
settle  = aBoard.settlementOverlay(zResult.terrain, c2f)
#print(names)
zTerr.hexGrid.builder.adjust("borders",borders)
zTerr.hexGrid.builder.adjust("names",names)
zTerr.hexGrid.builder.adjust("settle",settle)
#zTerr.hexGrid.builder.show()

I think we have enough pieces to get started on building the settlement map route. maybe clicking on the settlement would bring up a detail page we could put in either the right or left panel.

I think right panel and just an HTMX partial. This is a great example of when we are staying within a mode.

In [ ]:
??ZoomResult

In [ ]:
!tail -10 base.text

Any ideas how to fix showSettlement? can we add better logging?

So I think Settlement name + health/size stats would make sense. In theory we would have the list citzens on the left panel.

## Settlment Map

@rt("/settlement/{id}")
def settlementMap(session, id: str, rings: int = None):
    """Settlement map view — zoomed terrain with overlays."""
    uid = ensure_user(session)
    
    # Read from session if not provided, default 5
    if rings is None:
        rings = session.get('settlement_rings', 5)
    else:
        rings = max(3, min(12, rings))  # clamp
    session['settlement_rings'] = rings

    active = globalStore.active_board(uid)
    if not active:
        return RedirectResponse('/', status_code=303)

    try:
        place = globalStore.settlement_from_id(id)
        board = active.board
        terrain = board.terrain

        # Zoom into settlement region
        region = place.region(terrain.hexGrid, rings=rings)
        if len(region.hexes) == 0:
            return P("Settlement region is empty")

        result = active.cover.zoom_region_fast(region, compute_weather=True)
        zoomed = result.terrain
        zoomed.hexGrid.adjustRadius(terrain.hexGrid.radius)
        zoomed.hexGrid.builder.layers = []
        zoomed.colorMap()
        zoomed.hexGrid.update()

        c2f = result.invert_mapper()
        zoomed.terrainCream()

        builder = zoomed.hexGrid.builder
        builder.adjust("borders", board.countries_overlay(zoomed, c2f))
        builder.adjust("names", board.names_overlay(zoomed, c2f))

        if result.basins:
            builder.adjust("water", result.basins.draw_watersheds())

        # Settlement king piece — clickable, targets right panel
        settle_attrs = {
            'hx-get': f'/settlement_detail/{id}',
            'hx-target': '#debug-panel',
            'hx-swap': 'innerHTML',
            'style': 'cursor: pointer;',
        }
        settle_overlay = board.settlementTouchOverlay(zoomed, c2f, attrs=settle_attrs)
        builder.adjust("settlements", settle_overlay)

        # Kingdom info for breadcrumbs
        k = next((k for k in board.kingdoms if k.countryId == place.owner_id), None)
        kingdom_name = k.countryName if k else f"Kingdom {place.owner_id}"

        # --- Map div (primary swap target) ---
        map_div = Div(
            Div(
                A("🌍 World", hx_get="/showMap", hx_target="#map",
                  cls="btn btn-sm btn-outline"),
                Span(" → "),
                A(kingdom_name, hx_get=f"/kingdom/{place.owner_id}",
                  hx_target="#map", cls="btn btn-sm btn-outline"),
                Span(" → "),
                Span(place.name or "Settlement", cls="font-bold"),
                cls="flex items-center gap-2 p-2"
            ),
            HexTouchMap(zoomed.hexGrid, cls="w-full h-full"),
            id="map"
        )

        # --- Left panel (OOB swap) ---
        citizen_items = []
        for c in place.citizens:
            citizen_items.append(
                Li(A(f"🧑 {c.id[:8]}… HP:{c.health}/{c.max_health}",
                     hx_get=f"/piece_detail/{c.id}",
                     hx_target="#debug-panel",
                     cls="link link-hover"),
                   cls="text-sm")
            )
        if not citizen_items:
            citizen_items = [Li("No citizens yet", cls="text-sm opacity-50")]

        left_panel = Div(
            H4(place.name or "Settlement", cls="font-bold text-lg"),
            P(f"Kingdom: {kingdom_name}", cls=TextPresets.muted_sm),
            Divider(),
            P("Citizens", cls=TextPresets.bold_sm),
            Ul(*citizen_items, cls="space-y-1"),
            Divider(),
            A("← Back to Kingdom", hx_get=f"/kingdom/{place.owner_id}",
              hx_target="#map", cls="btn btn-outline btn-sm"),
            id="left-panel",
            hx_swap_oob="true",
            cls="w-64 bg-base-200 p-4 flex-shrink-0 overflow-y-auto"
        )

        return map_div, left_panel

    except Exception as e:
        logging.error(f"settlement route FAILED: {e}", exc_info=True)
        return P(f"Error loading settlement: {e}")


# ── Updated settlement_map — respects overlay toggles ────────

@rt("/settlement_map/{id}")
def settlement_map(session, id: str):
    """Map partial — renders zoomed terrain with active overlays only."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active: return P("No active game")

    rings = session.get('settlement_rings', 5)
    overlays = get_overlays(session)

    try:
        place = globalStore.settlement_from_id(id)
        board = active.board
        terrain = board.terrain

        region = place.region(terrain.hexGrid, rings=rings)
        if len(region.hexes) == 0:
            return P("Settlement region is empty", id="map")

        result = active.cover.zoom_region_fast(region, compute_weather=True)
        zoomed = result.terrain
        zoomed.hexGrid.adjustRadius(terrain.hexGrid.radius)
        zoomed.hexGrid.builder.layers = []
        zoomed.colorMap()
        zoomed.hexGrid.update()
        c2f = result.invert_mapper()

        builder = zoomed.hexGrid.builder

        # ── Conditional overlays ──
        if 'cream' in overlays:
            zoomed.terrainCream()

        builder.adjust("borders", board.countries_overlay(zoomed, c2f))  # always show borders

        if 'names' in overlays:
            builder.adjust("names", board.names_overlay(zoomed, c2f))

        if 'rivers' in overlays and result.basins:
            builder.adjust("water", result.basins.draw_watersheds())

        if 'settlements' in overlays:
            settle_attrs = {
                'hx-get': f'/settlement_detail/{id}',
                'hx-target': '#debug-panel',
                'hx-swap': 'innerHTML',
                'style': 'cursor: pointer;',
            }
            builder.adjust("settlements",
                           board.settlementTouchOverlay(zoomed, c2f, attrs=settle_attrs))

        return Div(
            HexTouchMap(zoomed.hexGrid, cls="w-full h-full"),
            id="map"
        )
    except Exception as e:
        logging.error(f"settlement_map FAILED: {e}", exc_info=True)
        return P(f"Error: {e}", id="map")


In [ ]:
@rt("/settlement_map/{id}")
def settlement_map(session, id: str, rings: int = None):
    """Map partial — renders zoomed terrain with active overlays."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return P("No active game", id="map")

    if rings is not None:
        rings = max(3, min(12, rings))
        session['settlement_rings'] = rings
    else:
        rings = session.get('settlement_rings', 5)

    overlays = get_overlays(session)

    try:
        place = globalStore.settlement_from_id(id)
        board = active.board
        terrain = board.terrain

        region = place.region(terrain.hexGrid, rings=rings)
        if len(region.hexes) == 0:
            return P("Settlement region is empty", id="map")

        result = active.cover.zoom_region_fast(region, compute_weather=True)
        zoomed = result.terrain
        zoomed.hexGrid.adjustRadius(terrain.hexGrid.radius)
        zoomed.hexGrid.builder.layers = []
        zoomed.colorMap()
        zoomed.hexGrid.update()
        c2f = result.invert_mapper()

        settle_attrs = {
            'hx-get': f'/settlement_detail/{id}',
            'hx-target': '#right-panel',
            'hx-swap': 'innerHTML',
            'style': 'cursor: pointer;',
        }
        zoomed.apply_overlays(result, board, c2f, overlays, settle_attrs)

        return Div(
            HexTouchMap(zoomed.hexGrid, cls="w-full h-full"),
            id="map"
        )
    except Exception as e:
        logging.error(f"settlement_map FAILED: {e}", exc_info=True)
        return P(f"Error: {e}", id="map")


### bottom panel

In [ ]:
# ── Controls route — HexLegend toggle bar ────────────────────

@rt("/settlement_controls/{id}")
def settlement_controls(session, id: str):
    """Bottom bar with toggleable overlay hex swatches."""
    active = get_overlays(session)

    hexes, values = [], []
    for key, cfg in OVERLAY_CONFIG.items():
        is_on = key in active
        fill   = cfg['fill']   if is_on else '#888'
        stroke = cfg['stroke'] if is_on else '#555'
        opacity = '1.0' if is_on else '0.35'

        style = StyleCSS(f"ov_{key}", fill=fill, stroke=stroke,
                         stroke_width=2, opacity=opacity)
        label = cfg['label'] + (' ✓' if is_on else '')
        h = Hex(radius=15, center=MapCord(0, 0), style=style, label=label)
        hexes.append(h)
        values.append(key)

    return HexLegend(
        hexes, name="overlay", values=values,
        hx_post=f"/toggle_overlay/{id}",
        hx_target="#map", hx_swap="outerHTML",
        direction="row", size=30, font_size=11,
        gap=4, item_gap=16,
    )


## On the side

@rt("/settlement/{id}")
def settlement(session, id: str):
    """Full-page shell for settlement view — panels load via HTMX."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return RedirectResponse('/', status_code=303)

    return Titled("Settlement",
        Div(
            # Left sidebar — breadcrumbs + citizens
            Div(
                id="left-panel",
                hx_get=f"/settlement_left/{id}",
                hx_trigger="load",
                hx_indicator="#spinner",
                cls="w-64 bg-base-200 p-4 flex-shrink-0 overflow-y-auto"
            ),

            # Main area
            Div(
                Loading(cls=(LoadingT.spinner, LoadingT.lg),
                        htmx_indicator=True, id="spinner"),
                Div(id="map",
                    hx_get=f"/settlement_map/{id}",
                    hx_trigger="load",
                    hx_indicator="#spinner",
                    cls="flex-1 overflow-auto"),
                Div(id="map-controls", cls="h-16 bg-base-300 p-2 flex-shrink-0"),
                cls="flex-1 flex flex-col items-center justify-center relative"
            ),

            # Right sidebar — detail card
            Div(
                H4("Settlement", cls="text-sm font-bold mb-2"),
                Div(id="debug-panel",
                    hx_get=f"/settlement_detail/{id}",
                    hx_trigger="load",
                    hx_indicator="#spinner"),
                cls="w-64 bg-base-200 p-4 flex-shrink-0 overflow-y-auto"
            ),
            cls="flex h-screen"
        )
    )


Do I need to wire settlement_controls into something? Should I redo def settlement(session, id: str):

In [ ]:
@rt("/settlement_left/{id}")
def settlement_left(session, id: str):
    """Left panel partial — breadcrumbs + citizen list."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return P("No active game")

    try:
        place = globalStore.settlement_from_id(id)
        board = active.board
        k = next((k for k in board.kingdoms if k.countryId == place.owner_id), None)
        kingdom_name = k.countryName if k else f"Kingdom {place.owner_id}"
        name = place.name or "Settlement"

        breadcrumbs = Div(
            A("🌍 World", hx_get="/showMap", hx_target="#map",
              cls="link link-hover text-sm"),
            Span(" → ", cls="opacity-50"),
            A(kingdom_name, hx_get=f"/kingdom/{place.owner_id}",
              hx_target="#map", cls="link link-hover text-sm"),
            Span(" → ", cls="opacity-50"),
            Span(name, cls="text-sm font-bold"),
            cls="flex flex-wrap items-center gap-1 mb-4"
        )

        citizen_items = []
        for c in getattr(place, 'citizens', []):
            citizen_items.append(
                Li(A(f"🧑 {c.id[:8]}… HP:{c.health}/{c.max_health}",
                     hx_get=f"/piece_detail/{c.id}",
                     hx_target="#right-panel",
                     cls="link link-hover"),
                   cls="text-sm")
            )
        if not citizen_items:
            citizen_items = [Li("No citizens yet", cls="text-sm opacity-50")]

        return Div(
            breadcrumbs,
            Divider(),
            H4(name, cls="font-bold text-lg"),
            P(f"Kingdom: {kingdom_name}", cls=TextPresets.muted_sm),
            Divider(),
            P("Citizens", cls=TextPresets.bold_sm),
            Ul(*citizen_items, cls="space-y-1"),
            Divider(),
            A("← Back to Kingdom",
              hx_get=f"/kingdom/{place.owner_id}",
              hx_target="#map", cls="btn btn-outline btn-sm"),
        )
    except Exception as e:
        logging.error(f"settlement_left FAILED: {e}", exc_info=True)
        return P(f"Error: {e}")


In [ ]:
def flag_hex(flag, radius=25):
    """Hex with flag pattern fill, built via SVGBuilder."""
    pad = 4
    size = radius * 2 + pad * 2
    center = MapCord(size / 2, size / 2)

    builder = SVGBuilder()
    builder.width = size
    builder.height = size

    # Register flag pattern as a fill
    pat_name = f"fh_{flag.name}"
    builder.add_definition(flag.flagPattern(pat_name, scale=0.08))
    pat_style = StyleCSS(pat_name, fill=f"url(#{pat_name})",
                         stroke=flag.comp, stroke_width=3)
    builder.add_style(pat_style)

    h = Hex(radius=radius, center=center, style=pat_style)
    builder.adjust("hex", h.svg())

    return Div(NotStr(builder.xml()))


In [ ]:
@rt("/settlement_detail/{id}")
def settlement_detail(session, id: str):
    """Right-panel summary card for a settlement."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return P("No active game")

    try:
        place = globalStore.settlement_from_id(id)
        k = next((k for k in active.board.kingdoms
                  if k.countryId == place.owner_id), None)
        kingdom_name = k.countryName if k else f"Kingdom {place.owner_id}"
        name = place.name or "Settlement"

        dark = k.flag.darkPrimary if k and k.flag else "#333"
        current_rings = session.get('settlement_rings', 5)

        header_parts = []
        if k and k.flag:
            header_parts.append(flag_hex(k.flag))
        header_parts.append(
            Div(
                H3(name, cls="font-bold",
                   style=f"font-family: 'Cinzel', serif; color: {dark};"),
                P(kingdom_name, cls=TextPresets.muted_sm,
                  style=f"font-family: 'Cinzel', serif; color: {dark}; opacity: 0.7;"),
            )
        )

        radius_control = Div(
            P("View Radius", cls=TextPresets.bold_sm),
            Input(type="range", name="rings", id="rings-slider",
                  min="3", max="12", value=str(current_rings),
                  hx_get=f"/settlement_map/{id}",
                  hx_target="#map",
                  hx_swap="outerHTML",
                  hx_trigger="change",
                  hx_include="#rings-slider",
                  cls="uk-range w-full"),
            P(f"{current_rings} rings", id="rings-label", cls=TextPresets.muted_sm),
            Script("""
                me('#rings-slider').on('input', ev => {
                    me('#rings-label').textContent = ev.target.value + ' rings';
                });
            """),
            cls="space-y-2"
        )

        return Div(
            Link(rel="stylesheet",
                 href="https://fonts.googleapis.com/css2?family=Cinzel:wght@400;700&display=swap"),
            DivLAligned(*header_parts, cls="gap-3"),
            Divider(),
            Div(
                DivFullySpaced(Span("Health", cls=TextPresets.muted_sm),
                               Span(f"{place.health}/100")),
                Progress(value=str(place.health), max="100"),
                DivFullySpaced(Span("Size", cls=TextPresets.muted_sm),
                               Span(str(place.size))),
                DivFullySpaced(Span("Citizens", cls=TextPresets.muted_sm),
                               Span(str(len(place.citizens)))),
                DivFullySpaced(Span("Location", cls=TextPresets.muted_sm),
                               Span(f"Hex {place.location}")),
                cls="space-y-2"
            ),
            Divider(),
            radius_control,
            cls="space-y-3"
        )
    except Exception as e:
        logging.error(f"settlement_detail FAILED: {e}", exc_info=True)
        return P(f"Error: {e}")


## Put it together

In [ ]:
@rt("/settlement/{id}")
def settlement(session, id: str):
    """Full-page shell for settlement view — panels load via HTMX."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return RedirectResponse('/', status_code=303)

    return Titled("Settlement",
        Div(
            # Left sidebar — breadcrumbs + citizens
            Div(
                id="left-panel",
                hx_get=f"/settlement_left/{id}",
                hx_trigger="load",
                hx_indicator="#spinner",
                cls="w-64 bg-base-200 p-4 flex-shrink-0 overflow-y-auto"
            ),

            # Main area
            Div(
                Loading(cls=(LoadingT.spinner, LoadingT.lg),
                        htmx_indicator=True, id="spinner"),
                Div(id="map",
                    hx_get=f"/settlement_map/{id}",
                    hx_trigger="load",
                    hx_indicator="#spinner",
                    cls="flex-1 overflow-auto"),
                # ← wired up here
                Div(id="map-controls",
                    hx_get=f"/settlement_controls/{id}",
                    hx_trigger="load",
                    cls="h-16 bg-base-300 p-2 flex-shrink-0"),
                cls="flex-1 flex flex-col items-center justify-center relative"
            ),

            # Right sidebar — detail card
            Div(
                H4("Settlement", cls="text-sm font-bold mb-2"),
                Div(id="right-panel",
                    hx_get=f"/settlement_detail/{id}",
                    hx_trigger="load",
                    hx_indicator="#spinner"),
                cls="w-64 bg-base-200 p-4 flex-shrink-0 overflow-y-auto"
            ),
            cls="flex h-screen"
        )
    )


In [ ]:
webMe(settlement_detail(dummySession, dummySettlment))

Did I build out everything correctly?

Did I build out everything correctly?

can you build what I am mising in settlement_detail, settlement_map, settlement_left and toggle_overlay 

Maybe we could have a slider that would set settlementdetail radius. We probably want to put this into a cookie for the session.

In [ ]:
webMe(settlement(dummySession, dummySettlment,rings = 2))

In [ ]:
??HexLegend

so I want to build out the map-controls part of our main setttlement page using a HexLegend at the bottom. We have several overlays that we would like to toggle settlement/names/cream/rivers that we should store in the cookie but then use when we are rendering the map.